# Module 1 - Foundations: Mapping the Terrain

**Time:** 09:30-11:00  
**Tool focus:** NetworkX

> We will see how data science powered by the SoBigData Research Infrastructure can help us investigate complex social systems.


## 1. Import Required Libraries

We'll import NetworkX for network analysis, Matplotlib for visualization, and additional libraries for data manipulation and analysis.


In [ ]:
# Import Required Libraries
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import Counter
import seaborn as sns

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# IGNORE THIS :) 
def build_demo_graph(seed=42):
    rng = np.random.default_rng(seed)
    communities = [
        {"label": "left-mainstream", "camp": "left", "size": 36, "mean_opinion": -0.55, "enclave": 0},
        {"label": "left-enclave", "camp": "left", "size": 18, "mean_opinion": -0.88, "enclave": 1},
        {"label": "right-mainstream", "camp": "right", "size": 36, "mean_opinion": 0.55, "enclave": 0},
        {"label": "right-enclave", "camp": "right", "size": 18, "mean_opinion": 0.88, "enclave": 1},
    ]
    sizes = [community["size"] for community in communities]
    probability_matrix = [
        [0.18, 0.08, 0.02, 0.004],
        [0.08, 0.25, 0.006, 0.001],
        [0.02, 0.006, 0.18, 0.08],
        [0.004, 0.001, 0.08, 0.25],
    ]

    graph = nx.stochastic_block_model(sizes, probability_matrix, seed=seed)
    graph.graph.clear()

    node_id = 0
    for community in communities:
        for _ in range(community["size"]):
            graph.nodes[node_id]["label"] = f"user_{node_id:03d}"
            graph.nodes[node_id]["community_label"] = community["label"]
            graph.nodes[node_id]["camp"] = community["camp"]
            graph.nodes[node_id]["opinion"] = float(np.clip(rng.normal(community["mean_opinion"], 0.09), -1.0, 1.0))
            graph.nodes[node_id]["enclave"] = int(community["enclave"])
            graph.nodes[node_id]["activity"] = int(rng.poisson(9 if community["enclave"] else 6) + 1)
            node_id += 1

    return graph

## 2. Build and Visualize the network

We'll build a synthetic graph

In [ ]:
# Build graph 
G = build_demo_graph()

In [ ]:
print("Network Summary (demo graph):")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")
print(f"Connected components: {nx.number_connected_components(G)}")

In [ ]:
# Visualize the Sawmill network
plt.figure(figsize=(14, 10))

pos = nx.spring_layout(G, seed=42)
node_sizes = [G.degree(node) * 150 + 300 for node in G.nodes()]

nx.draw(G, pos, 
    node_size=node_sizes,
    node_color='#FF6B6B',
    edge_color='#999999',
    alpha=0.9,
    width=1.5,
    with_labels=True,
    font_size=8,
    font_weight='bold')

plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
deg = [d for _, d in G.degree()]
dist = Counter(deg) 
dist # keys are degree values, values are n. of nodes having that degree

plt.loglog(dist.keys(), dist.values(), "o")
plt.xlabel("Degree k")
plt.ylabel("Number of nodes with degree k")
plt.title("Degree distribution")
plt.show()


## 3. Calculate Basic Network Measures

These metrics summarize the structure of the Sawmill network. In this context:
- **Density** reflects how tightly connected the employees are.
- **Clustering** suggests local grouping or trust clusters.
- **Path length** and **diameter** indicate how many steps separate actors (reachability across the network).


In [ ]:
# Calculate basic network measures

# 1. Density: proportion of possible connections that exist
density = nx.density(G)
print(f"Network Density: {density:.4f}")
print(f"  → Interpretation: {density*100:.2f}% of all possible connections exist")


In [ ]:
# 2. Diameter: longest shortest path between any two nodes
if nx.is_connected(G):
    diameter = nx.diameter(G)
    print(f"\nNetwork Diameter: {diameter}")
    print(f"  → Interpretation: Maximum distance between any two actors is {diameter} steps")
else:
    print("\nNetwork is disconnected. Calculating diameter for largest component:")
    largest_cc = max(nx.connected_components(G), key=len)
    subgraph = G.subgraph(largest_cc)
    diameter = nx.diameter(subgraph)
    print(f"Diameter of largest component: {diameter}")


In [ ]:

# 3. Average Clustering Coefficient: tendency to form triangles/cliques
avg_clustering = nx.average_clustering(G)
print(f"\nAverage Clustering Coefficient: {avg_clustering:.4f}")
print(f"  → Interpretation: Likelihood of forming tight-knit groups: {avg_clustering*100:.2f}%")


In [ ]:
# 4. Average Shortest Path Length
if nx.is_connected(G):
    avg_path_length = nx.average_shortest_path_length(G)
    print(f"\nAverage Shortest Path Length: {avg_path_length:.4f}")
    print(f"  → Interpretation: Average distance between any two actors is {avg_path_length:.2f} steps")
else:
    largest_cc = max(nx.connected_components(G), key=len)
    subgraph = G.subgraph(largest_cc)
    avg_path_length = nx.average_shortest_path_length(subgraph)
    print(f"\nAverage Shortest Path Length (largest component): {avg_path_length:.4f}")


In [ ]:
# 5. Number of connected components
num_components = nx.number_connected_components(G)
print(f"\nNumber of Connected Components: {num_components}")
print(f"  → Interpretation: Network is divided into {num_components} separate group(s)")


In [ ]:
# Create a summary table of basic measures
measures_summary = pd.DataFrame({
    'Measure': ['Density', 'Diameter', 'Avg Clustering Coefficient', 'Avg Path Length', 'Connected Components'],
    'Value': [
        f"{density:.4f}",
        f"{diameter}",
        f"{avg_clustering:.4f}",
        f"{avg_path_length:.4f}",
        f"{num_components}"
    ],
    'Interpretation': [
        'Proportion of possible ties that exist',
        'Maximum distance between actors',
        'Tendency to form local clusters (0-1)',
        'Average steps between actors',
        'Number of separate groups'
    ]
})

print("\nNetwork Measures Summary ( dataset:")
print(measures_summary.to_string(index=False))